# Amazon Ads Analytics — Business Analysis

Walkthrough of the SQL warehouse. Each section runs one analytical query
against the warehouse, shows results, charts them, and discusses what
they mean operationally.

**Data context:** four Amazon marketplace reports for a single brand
("Brand_A") over a ~one-month window, anonymized programmatically.


## Setup

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.2f}".format)

DB = Path("../data/warehouse.db")
QUERIES = Path("../sql/queries")

conn = sqlite3.connect(DB)

def run(sql_file):
    sql = (QUERIES / sql_file).read_text()
    return pd.read_sql_query(sql, conn)


### Warehouse sanity check

In [ ]:
for table in ["dim_date", "dim_campaign", "dim_product",
              "fact_ads_daily", "fact_search_term", "fact_business_report"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<25} {n:>7,} rows")


## Q1 — Campaign ROAS with TACoS context

**Question:** which campaigns are most efficient on their own ROAS, and how do
they look against TOTAL business sales (TACoS lens)?

**Why it matters:** a campaign can have great ROAS in isolation but
contribute a tiny share of business sales. Looking at both lenses
prevents over-celebrating small efficient campaigns at the expense of
portfolio decisions.


In [ ]:
q1 = run("01_campaign_roas_tacos.sql")
print(f"Campaigns analyzed: {len(q1):,}")
q1.head(10)


In [ ]:
active = q1[q1["spend"] >= 50].copy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(active["roas"], bins=30, color="#3B82F6", edgecolor="white")
ax.axvline(active["roas"].median(), color="#DC2626", linestyle="--",
           label=f"median ROAS = {active['roas'].median():.2f}")
ax.set_xlabel("ROAS")
ax.set_ylabel("Campaign count")
ax.set_title("ROAS distribution (campaigns with spend >= $50)")
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation (fill in based on results):**

- Top performing campaigns by ROAS: _list 2-3_
- Portfolio-level TACoS: _X%_. Under 15% is generally healthy.
- _Add observations on which campaigns are over- or under-funded._


## Q2 — Harvestable search terms

**Question:** which broad/auto search terms have converted profitably
and should be promoted to exact-match campaigns?

**Why it matters:** keyword harvesting is one of the highest-ROI
activities in Amazon Ads management.


In [ ]:
q2 = run("02_harvestable_search_terms.sql")
print(f"Harvestable terms: {len(q2):,}")
q2.head(15)


In [ ]:
if len(q2) > 0:
    top = q2.nlargest(15, "sales").iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top["customer_search_term"].str[:50], top["sales"], color="#10B981")
    ax.set_xlabel("Attributed sales (7-day)")
    ax.set_title("Top 15 harvestable search terms by sales")
    plt.tight_layout()
    plt.show()
else:
    print("No harvestable terms found with current thresholds.")


**Interpretation:**

- _Which terms surprised you?_
- _Which would you promote first?_


## Q3 — Wasted spend detection

**Question:** which search terms eat budget without converting?
Candidates for negative keywords.


In [ ]:
q3 = run("03_wasted_spend.sql")
print(f"Wasted-spend candidates: {len(q3):,}")
print(f"Total wasted spend: ${q3['spend'].sum():,.2f}")
q3.head(15)


In [ ]:
if len(q3) > 0:
    by_reason = q3.groupby("waste_reason")["spend"].agg(["sum", "count"]).sort_values("sum", ascending=False)
    by_reason.columns = ["total_spend", "n_search_terms"]
    print(by_reason)


**Interpretation:**

- _Total wasted: $X over the period_
- _Most common waste reason: ..._
- _Specific search terms to negativize now: ..._


## Q4 — Ads vs organic synergy at the ASIN level

**Question:** what share of sales is ads vs organic per product?

**Why it matters:** foundational view for marketing-mix thinking on Amazon.


In [ ]:
q4 = run("04_ads_vs_organic_synergy.sql")
print(f"ASINs analyzed: {len(q4):,}")
q4.head(15)


In [ ]:
if len(q4) > 0:
    health_dist = q4["health_label"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    health_dist.plot(kind="bar", color="#F59E0B", ax=ax)
    ax.set_title("ASIN health-label distribution (ads vs organic)")
    ax.set_xlabel("Health label")
    ax.set_ylabel("ASINs")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print(health_dist)


**Interpretation & data quality note:**

> **Observed mismatch:** Business Report uses Parent ASINs while
> Advertised Product Report uses Child ASINs. Some ASINs show "no_ads"
> when they actually are advertised. In production, fix with a
> parent-child ASIN dimension from Amazon's catalog.

- _Which products are most ad-dependent?_
- _Which have strong organic and could reduce ad spend?_


## Q5 — Daily trend with rolling averages

**Question:** is portfolio performance trending up, down, or stable?
Where are the inflection points?


In [ ]:
q5 = run("05_daily_trend_anomalies.sql")
q5["date_id"] = pd.to_datetime(q5["date_id"])
print(f"Days analyzed: {len(q5)}")
q5.head()


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(q5["date_id"], q5["daily_spend"], color="#3B82F6", alpha=0.4, label="daily")
ax1.plot(q5["date_id"], q5["rolling_7d_spend"], color="#1E40AF", linewidth=2, label="7-day rolling")
ax1.set_ylabel("Spend ($)")
ax1.set_title("Daily ad spend with 7-day rolling average")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(q5["date_id"], q5["daily_roas"], color="#10B981", alpha=0.4, label="daily")
ax2.plot(q5["date_id"], q5["rolling_7d_roas"], color="#047857", linewidth=2, label="7-day rolling")
anomalies = q5[q5["anomaly_flag"] == "roas_anomaly"]
if len(anomalies) > 0:
    ax2.scatter(anomalies["date_id"], anomalies["daily_roas"],
                color="#DC2626", s=60, zorder=5, label="anomaly")
ax2.set_ylabel("ROAS")
ax2.set_xlabel("Date")
ax2.set_title("Daily ROAS with anomaly detection")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Interpretation:**

- _Trend over the window: ..._
- _Anomaly days observed: ..._
- _Probable causes to investigate: ..._


## Summary & data quality observations

**Pipeline outputs:**
1. Portfolio ROAS / TACoS overview (Q1)
2. Harvestable search terms ready for promotion (Q2)
3. Wasted-spend candidates for negative keywords (Q3)
4. Per-ASIN ads-vs-organic health labels (Q4)
5. Daily monitoring with anomaly flags (Q5)

**Data quality issues encountered:**
- **ASIN granularity mismatch:** Business Report uses Parent ASINs, Advertised
  Product Report uses Child ASINs. Joining them directly under-reports ad
  contribution.
- **Reporting window heterogeneity:** Business Report is a snapshot,
  ad reports are daily.
- **Currency formatting:** dollar strings with `$` and commas need cleaning
  before numeric coercion.

**Improvements for v2:**
- Wire orchestration (Prefect or Airflow) for daily runs.
- Add Great Expectations checks to fail loudly on schema or value drift.
- Add a parent-child ASIN dimension to resolve the ads-vs-organic join.
- Migrate from SQLite to a managed warehouse if scale grows.


In [ ]:
conn.close()